# Day 3 - Homework: Market basket on real transactions

**Goal:** run the exact technique from Lab 3 on data that is *natively* transactional, one row per (basket, item), and see how much stronger the rules become. This is market basket analysis working with the grain of the data rather than against it.

Keep Lab 3 in mind: there, the strongest rule from constructed portal baskets barely reached a lift of 1.1. Watch what happens here.

In [1]:
import importlib.util
if importlib.util.find_spec('mlxtend') is None:
    raise RuntimeError("mlxtend is missing. In your activated .venv run:  "
                       "pip install -r requirements.txt   then restart the kernel.")

import pandas as pd
from pathlib import Path

def find_data(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / 'data').is_dir():
            return p / 'data'
    raise FileNotFoundError('data/ folder not found')

DATA = find_data()
tx = pd.read_csv(DATA / 'retail' / 'retail_transactions.csv')
print('rows (basket-item pairs):', len(tx))
print('baskets:', tx['basket_id'].nunique(), ' distinct items:', tx['item'].nunique())
tx.head()

rows (basket-item pairs): 20197
baskets: 4000  distinct items: 15


,basket_id,item
0,1,Bananas
1,1,Bread
2,1,Butter
3,1,Cheese
4,1,Crisps


## 1. Shape into transactions

The file is long form: one row per item in a basket. Market basket wants each basket as a *set* of items.

In [2]:
baskets = tx.groupby('basket_id')['item'].apply(list)
print('example basket:', baskets.iloc[0])
print('average basket size:', baskets.apply(len).mean().round(1))

example basket: ['Bananas', 'Bread', 'Butter', 'Cheese', 'Crisps', 'Pasta', 'Soft Drinks']
average basket size: 5.0


## 2. Frequent itemsets and rules

In [3]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

te = TransactionEncoder()
enc = pd.DataFrame(te.fit_transform(baskets.tolist()), columns=te.columns_)
freq = apriori(enc, min_support=0.03, use_colnames=True)
rules = association_rules(freq, metric='lift', min_threshold=1.0).sort_values('lift', ascending=False)
print(f'frequent itemsets: {len(freq)}   rules: {len(rules)}')

frequent itemsets: 470   rules: 1034


## 3. Read the strongest rules

Three numbers describe every rule. **Support**: how common the combination is overall. **Confidence**: given the left-hand items, how often the right-hand items appear. **Lift**: how many times more likely that is than chance. Lift above 1 means a genuine association; the higher, the stronger.

In [4]:
top = rules.head(8).copy()
top['antecedents'] = top['antecedents'].apply(lambda s: ', '.join(s))
top['consequents'] = top['consequents'].apply(lambda s: ', '.join(s))
print(top[['antecedents', 'consequents', 'support', 'confidence', 'lift']].to_string(index=False))

             antecedents              consequents  support  confidence     lift
Tomato Sauce, Baby Wipes           Pasta, Nappies  0.07275    0.447005 4.267347
          Pasta, Nappies Tomato Sauce, Baby Wipes  0.07275    0.694511 4.267347
          Sugar, Nappies       Baby Wipes, Coffee  0.06000    0.524017 4.134260
      Baby Wipes, Coffee           Sugar, Nappies  0.06000    0.473373 4.134260
         Coffee, Nappies        Sugar, Baby Wipes  0.06000    0.571429 4.081633
       Sugar, Baby Wipes          Coffee, Nappies  0.06000    0.428571 4.081633
       Pasta, Baby Wipes    Tomato Sauce, Nappies  0.07275    0.527174 4.001320
   Tomato Sauce, Nappies        Pasta, Baby Wipes  0.07275    0.552182 4.001320


These lifts sit far above 1, several times stronger than anything the portal baskets could produce. Same algorithm, same three lines of mlxtend. The only thing that changed is that this data is *actually* transactional. That contrast is the whole point of the exercise.

## 4. From rule to decision

A strong rule is not automatically a useful one. The skill market basket analysis really trains is asking the right business question of a rule: is it *actionable*, and is it *causal enough to act on*?

Retail folklore is full of 'surprising' pairings, two apparently unrelated products that supposedly sell together, stories retold as tidy causal wins when they were nothing of the sort. That is the right instinct to carry: a high lift tells you two things co-occur, not *why*. Before acting, ask whether the association would survive being turned into a shelf layout, a bundle or a recommendation, or whether it is just two popular items that happen to share baskets.

## Your turn

1. Pick the single rule you would actually act on if you ran this shop, and justify it using support, confidence and lift together rather than lift alone.
2. Raise `min_support` to 0.10 and rerun. Which rules survive, and what kind of association does a high support threshold bias you towards?
3. In two sentences, explain to a non-technical manager why the portal 'baskets' in Lab 3 gave such weak rules while this dataset gives strong ones.

In [5]:
# your turn
